
# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [ ]:
# Partie I. Configuration (à exécuter une seule fois)
# Installation des dépendances minimales ; mode silencieux pour réduire le bruit.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True


### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [ ]:
import pandas as pd
from datasets import load_dataset

# Chemin vers vos données ; laissez vide pour utiliser l'échantillon HF cnn_dailymail ou le fallback
train_path = ''  # ex: '/content/train.csv'
test_path = ''   # ex: '/content/test.csv'

# Données de secours en cas d'échec de téléchargement
fallback = pd.DataFrame([
    {
        'prompt_text': 'Le chat s\'est assis sur le tapis et a ronronné bruyamment pendant que le soleil se couchait.',
        'prompt_title': 'Chat se repose sur un tapis au coucher du soleil'
    },
    {
        'prompt_text': 'Des scientifiques ont découvert de l\'eau sur la lune, ouvrant de nouvelles voies de recherche.',
        'prompt_title': 'De l\'eau trouvée sur la lune'
    },
    {
        'prompt_text': 'L\'équipe locale a remporté le championnat après un match final dramatique.',
        'prompt_title': 'L\'équipe locale décroche le titre'
    },
])

def load_and_sample(path, split_name, n):
    """
    Charge les données depuis un CSV local ou depuis Hugging Face.
    """
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"Échec du chargement HF ({exc}); utilisation de l\'échantillon de secours.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [3]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List
import pandas as pd

def batch_generator(items: List[str], batch_size: int):
    """Générateur pour diviser une liste en lots (batches)."""
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    """
    Génère des résumés en utilisant T5.
    - Ajoute le préfixe 'summarize: ' requis par T5.
    - Gère le passage sur GPU si disponible.
    - Nettoie la mémoire cache pour éviter les erreurs de saturation (OOM).
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Utilisation du périphérique: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    all_summaries = []

    # Traitement par lots pour l'efficacité
    for batch in batch_generator(texts, batch_size):
        # T5 nécessite ce préfixe spécifique pour savoir qu'il doit résumer
        inputs = ["summarize: " + doc for doc in batch]

        # Encodage avec troncature pour ne pas dépasser la limite du modèle
        inputs_encoded = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

        # Génération des IDs de sortie
        outputs = model.generate(
            inputs_encoded["input_ids"],
            max_new_tokens=max_new_tokens,
            num_beams=2,
            early_stopping=True
        )

        # Décodage pour transformer les IDs en texte lisible
        batch_preds = [tokenizer.decode(g, skip_special_tokens=True, clean_up_tokenization_spaces=True) for g in outputs]
        all_summaries.extend(batch_preds)

        # Libération de la mémoire GPU
        if device == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()

    return all_summaries

# Activation du flag pour tester l'implémentation
RUN_T5 = True
if RUN_T5:
    # On teste sur un petit échantillon pour gagner du temps
    sample_size = 5
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].iloc[:sample_size].tolist(), model_name='t5-small', batch_size=2)

    results_df = pd.DataFrame({
        'Texte Original': train_df['prompt_text'].iloc[:sample_size],
        'Résumé Référence': train_df['prompt_title'].iloc[:sample_size],
        'Résumé T5 (Généré)': train_summaries_t5
    })
    display(results_df)

Utilisation du périphérique: cuda


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

,Texte Original,Résumé Référence,Résumé T5 (Généré)
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,the toys contain a chemical that converts into...
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,president pervez Musharraf orders troops to ta...
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko will face comeback ...



### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [ ]:
from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    """
    Calcule l'exactitude naïve par correspondance exacte des chaînes.
    """
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].iloc[:len(train_summaries_t5)].tolist())
    print(f"Exactitude (correspondance exacte) : {acc:.4f}")
else:
    print("Calcul de l'exactitude sauté (pas de prédictions).")


### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [4]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

# Chargement de la métrique ROUGE
rouge = evaluate.load('rouge')

def normalize_text(text: str) -> str:
    """
    Prépare le texte pour ROUGE :
    - Tokenise en phrases
    - Rejoint avec des sauts de ligne (standard pour ROUGE-L)
    """
    sents = sent_tokenize(text.strip())
    return "\n".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    """
    Calcule les scores ROUGE-1, ROUGE-2 et ROUGE-L.
    """
    # Normalisation des prédictions et des références
    decoded_preds = [normalize_text(p) for p in preds]
    decoded_labels = [normalize_text(r) for r in refs]

    # Calcul effectif avec la bibliothèque evaluate
    result = rouge.compute(predictions=decoded_preds,
                           references=decoded_labels,
                           use_stemmer=True)

    return result

# Test de validation (Sanity Check)
test_preds = ["alpha beta", "", "The cat sat on the mat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]

print("Vérification des scores ROUGE :")
results = compute_rouge_score(test_preds, test_refs)
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

Vérification des scores ROUGE :
rouge1: 0.5556
rouge2: 0.5238
rougeL: 0.5556
rougeLsum: 0.5556



### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.



### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import pandas as pd

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    """
    Génère des résumés avec GPT-2 en utilisant le prompt 'TL;DR:'.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    all_summaries = []
    for batch in batch_generator(texts, batch_size):
        inputs = [text[:1000] + "\n\nTL;DR:" for text in batch]
        inputs_encoded = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)

        outputs = model.generate(
            inputs_encoded["input_ids"],
            attention_mask=inputs_encoded["attention_mask"],
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )

        for output in outputs:
            decoded = tokenizer.decode(output, skip_special_tokens=True)
            summary = decoded.split("TL;DR:")[-1].strip()
            all_summaries.append(summary)

        if device == 'cuda':
            torch.cuda.empty_cache()

    return all_summaries

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'Reference'):
    """Calcule ROUGE-L par ligne."""
    scores = []
    for _, row in df.iterrows():
        if not row[pred_col] or str(row[pred_col]).strip() == "":
            scores.append(0.0)
        else:
            res = compute_rouge_score([row[pred_col]], [row[ref_col]])
            scores.append(res['rougeL'])
    return scores

# Exécution et calcul des scores
RUN_COMPARE = True
if RUN_COMPARE:
    print("Génération des résumés et calcul des scores...")
    sample_texts = train_df['prompt_text'].iloc[:5].tolist()
    gpt2_summaries = summarize_with_gpt2(sample_texts)

    comparison_df = pd.DataFrame({
        'Reference': train_df['prompt_title'].iloc[:5],
        'T5_Summary': train_summaries_t5[:5], # S'assure de la même taille
        'GPT2_Summary': gpt2_summaries
    })

    # Calcul explicite des colonnes de score
    comparison_df['ROUGE_L_T5'] = compute_rouge_per_row(comparison_df, 'T5_Summary')
    comparison_df['ROUGE_L_GPT2'] = compute_rouge_per_row(comparison_df, 'GPT2_Summary')

    display(comparison_df.head())

Génération des résumés et calcul des scores...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


,Reference,T5_Summary,GPT2_Summary,ROUGE_L_T5,ROUGE_L_GPT2
0,Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,", the world championship race is over.\n\n\nHa...",0.215385,0.126984
1,State-run news agency: China orders an investi...,the toys contain a chemical that converts into...,The Aqua Dots toys contain a chemical that can...,0.144928,0.108108
2,The company has become a huge name in communic...,,Qualcomm is a big company. It's a big company....,0.000000,0.176471
3,NEW: President Musharraf orders troops to take...,president pervez Musharraf orders troops to ta...,", Musharraf's actions are a response to the gr...",0.380952,0.202899
4,Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko will face comeback ...,Julia Vakulenko will face comeback queen Linds...,0.310345,0.305085



### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [9]:
import pandas as pd

def compare_models(rouge_results_dict):
    """
    Prend un dictionnaire de type {nom_modele: liste_de_scores_rouge_L}
    et retourne un DataFrame avec les moyennes.
    """
    summary_data = []
    for model_name, scores in rouge_results_dict.items():
        avg_score = sum(scores) / len(scores) if scores else 0
        summary_data.append({"Modèle": model_name, "ROUGE-L Moyen": round(avg_score, 4)})

    return pd.DataFrame(summary_data)

def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    """
    Affiche les résumés côte à côte pour une comparaison visuelle.
    """
    cols_to_show = ['Reference'] + pred_cols
    existing_cols = [c for c in cols_to_show if c in df.columns]
    return df[existing_cols]

# Synthèse finale
if 'comparison_df' in locals():
    print("--- Synthèse des Performances ---")

    stats_dict = {}
    if 'ROUGE_L_T5' in comparison_df.columns:
        stats_dict["T5-Small"] = comparison_df['ROUGE_L_T5'].tolist()
    if 'ROUGE_L_GPT2' in comparison_df.columns:
        stats_dict["GPT-2"] = comparison_df['ROUGE_L_GPT2'].tolist()

    if stats_dict:
        stats_df = compare_models(stats_dict)
        display(stats_df)
    else:
        print("Aucun score ROUGE trouvé. Relancez la cellule de la Partie VII.")

    print("\n--- Comparaison Qualitative ---")
    preds = [c for c in ['T5_Summary', 'GPT2_Summary'] if c in comparison_df.columns]
    qualitative_df = compare_models_summaries(comparison_df, preds)
    display(qualitative_df)
else:
    print("Erreur : comparison_df n'est pas défini.")

--- Synthèse des Performances ---


,Modèle,ROUGE-L Moyen
0,T5-Small,0.2103
1,GPT-2,0.1839



--- Comparaison Qualitative ---


,Reference,T5_Summary,GPT2_Summary
0,Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,", the world championship race is over.\n\n\nHa..."
1,State-run news agency: China orders an investi...,the toys contain a chemical that converts into...,The Aqua Dots toys contain a chemical that can...
2,The company has become a huge name in communic...,,Qualcomm is a big company. It's a big company....
3,NEW: President Musharraf orders troops to take...,president pervez Musharraf orders troops to ta...,", Musharraf's actions are a response to the gr..."
4,Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko will face comeback ...,Julia Vakulenko will face comeback queen Linds...


## Synthèse et Réflexions

### Réponses aux questions :

1. **Quelles métriques sont les plus informatives ?**
   - Le score **ROUGE-L** est bien plus informatif que l'exactitude (accuracy). Il mesure la plus longue sous-séquence commune, ce qui permet de capturer la fluidité et la structure du résumé plutôt que de simples correspondances de mots isolés.

2. **Impact de la taille du modèle sur ROUGE et la qualité qualitative ?**
   - Un modèle plus grand (comme T5-Base par rapport à T5-Small) capture mieux les nuances contextuelles. Qualitativement, on observe que T5 produit des résumés plus structurés, tandis que GPT-2 (sans fine-tuning) a tendance à être plus répétitif (comme vu avec l'exemple 'Qualcomm').

3. **Pourquoi l'exactitude (accuracy) échoue-t-elle ici ?**
   - Dans la génération de langage naturel, il existe une infinité de façons de résumer correctement un texte. L'exactitude brute exige une correspondance mot pour mot, ce qui est statistiquement presque impossible, rendant ce score inutilement sévère (proche de zéro).

4. **Extensions possibles ?**
   - On pourrait ajouter une **évaluation humaine** avec une échelle de Likert (cohérence, pertinence, factualité) ou utiliser des métriques basées sur l'encodage comme **BERTScore** pour capturer la similarité sémantique plutôt que syntaxique.

### Conclusion
Cet exercice illustre l'importance de choisir une métrique adaptée à la tâche. Alors que T5 est entraîné spécifiquement avec un objectif de 'text-to-text', GPT-2 nécessite des techniques de prompting plus avancées (ou un fine-tuning) pour atteindre des performances compétitives en résumé.